In [20]:
import pandas as pd
import json

businesses = []
with open('data/raw/yelp_academic_dataset_business.json', encoding='utf-8') as f:
    for line in f:
        businesses.append(json.loads(line))

business_df = pd.DataFrame(businesses)
business_df.shape

(150346, 14)

In [21]:
# Check which cities have the most businesses in this dataset
business_df['city'].value_counts().head(20)

city
Philadelphia        14569
Tucson               9250
Tampa                9050
Indianapolis         7540
Nashville            6971
New Orleans          6209
Reno                 5935
Edmonton             5054
Saint Louis          4827
Santa Barbara        3829
Boise                2937
Clearwater           2221
Saint Petersburg     1663
Metairie             1643
Sparks               1624
Wilmington           1446
Franklin             1414
St. Louis            1255
St. Petersburg       1185
Meridian             1043
Name: count, dtype: int64

In [22]:
# Filter businesses to just Philadelphia
philly_business = business_df[business_df['city'] == 'Philadelphia'].copy()
philly_business.shape

(14569, 14)

In [23]:
philly_business_ids = set(philly_business['business_id'])
# Get Philly reviews
philly_reviews = []
with open('data/raw/yelp_academic_dataset_review.json', encoding='utf-8') as f:
    for line in f:
        review = json.loads(line)
        if review['business_id'] in philly_business_ids:
            philly_reviews.append(review)

philly_reviews_df = pd.DataFrame(philly_reviews)

# Get Philly users
philly_user_ids = set(philly_reviews_df['user_id'])
philly_users = []
with open('data/raw/yelp_academic_dataset_user.json', encoding='utf-8') as f:
    for line in f:
        user = json.loads(line)
        if user['user_id'] in philly_user_ids:
            philly_users.append(user)

philly_users_df = pd.DataFrame(philly_users)
philly_users_df.shape


(279855, 22)

In [24]:
# Convert nested dict/list columns to JSON strings so SQLite can store them
philly_business['attributes'] = philly_business['attributes'].apply(lambda x: json.dumps(x) if x is not None else None)
philly_business['hours'] = philly_business['hours'].apply(lambda x: json.dumps(x) if x is not None else None)

In [25]:
conn = sqlite3.connect('data/philly_yelp.db')

philly_business.to_sql('business', conn, if_exists='replace', index=False)
philly_reviews_df.to_sql('review', conn, if_exists='replace', index=False)
philly_users_df.to_sql('user', conn, if_exists='replace', index=False)

conn.close()

print("Database created successfully")

Database created successfully
